# 1. Environment

In [20]:
from urllib.parse import quote

import time
import re

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd

In [2]:
ROOT = Path(r"C:\Users\yatsyk\Documents\NextCloud_Synchro\passim_BoH")
CSV_PATH = ROOT / "04.CoMMa" / "02.metadata" / "bbma_is+arkindex_id.csv"
BACKUP_PATH = CSV_PATH.with_suffix(".before_comma_p_lookup.bak")
REQUEST_DELAY_SECONDS = 0.2
TIMEOUT_SECONDS = 20

CSV_PATH

WindowsPath('C:/Users/yatsyk/Documents/NextCloud_Synchro/passim_BoH/04.CoMMa/02.metadata/bbma_is+arkindex_id.csv')

# 2. Data collection

## Load the TSV exported from heurist

There are 324 mss in Heurist database with both Arkindex ID and bbma_data_entity.
https://heurist.huma-num.fr/heurist/?db=stutzmann_horae&w=a&q=%5B%7B%22t%22%3A%2253%22%7D%2C%7B%22f%3A1177%22%3A%22%22%7D%2C%7B%22f%3A1258%22%3A%22%22%7D%2C%7B%22sortby%22%3A%22t%22%7D%5D

In [ ]:
df = pd.read_csv(CSV_PATH, sep="\t", dtype=str).fillna("")

target_mask = (
    (df["URL CoMMA"].str.strip() == "NC") &
    df["bbma_data_entity"].str.strip().str.contains(r"/Q\d+$", regex=True)
)

print(f"Rows in file: {len(df)}")
print(f"Rows to check: {target_mask.sum()}")

df.loc[target_mask, ["Manuscript H-ID", "bbma_data_entity", "URL CoMMA"]]

Rows in file: 324
Rows to check: 230


,Manuscript H-ID,bbma_data_entity,URL CoMMA
0,349,https://data.biblissima.fr/entity/Q202618,NC
1,350,https://data.biblissima.fr/entity/Q202641,NC
2,352,https://data.biblissima.fr/entity/Q202515,NC
3,353,https://data.biblissima.fr/entity/Q202568,NC
4,524,https://data.biblissima.fr/entity/Q218093,NC


## Helper functions

For mss with URL CoMMA == NC hese functions
1. extract bbma_data_entity
2. use Selenium to check whether a correspondant CoMMa document pages exists, and, if this is the case,
3. replace NC with correct link.

In [21]:
def build_entity_doc_url(bbma_data_entity: str) -> str:
    encoded_once = quote(bbma_data_entity.strip(), safe="")
    encoded_twice = quote(encoded_once, safe="")
    return f"https://comma.inria.fr/doc/{encoded_twice}"


def make_driver(headless: bool = True):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,1000")
    return webdriver.Chrome(options=options)


def find_first_p_page_selenium(driver, bbma_data_entity: str, timeout: int = 30) -> str | None:
    entity_doc_url = build_entity_doc_url(bbma_data_entity)
    driver.get(entity_doc_url)

    try:
        WebDriverWait(driver, timeout).until(
            lambda d: "/p/" in d.current_url or "/p/" in d.page_source
        )
    except Exception:
        return None

    # если CoMMa сам открыл первую страницу
    if "/p/" in driver.current_url:
        return driver.current_url

    html = driver.page_source

    matches = re.findall(
        r'https://comma\.inria\.fr/doc/[^"\'<>\s]+/p/[^"\'<>\s]+',
        html
    )

    matching_urls = [
        url for url in matches
        if url.startswith(entity_doc_url + "/p/")
    ]

    if not matching_urls:
        return None

    return sorted(set(matching_urls))[0]

In [23]:
updates = {}

driver = make_driver(headless=True)

try:
    for idx, row in df.loc[target_mask].iterrows():
        bbma_data_entity = row["bbma_data_entity"]

        p_url = find_first_p_page_selenium(driver, bbma_data_entity)

        if p_url:
            updates[idx] = p_url
            print(f"[OK] {bbma_data_entity} -> {p_url}")
        else:
            print(f"[MISS] {bbma_data_entity}")

        time.sleep(REQUEST_DELAY_SECONDS)

finally:
    driver.quit()

print(f"Rows checked: {target_mask.sum()}")
print(f"Rows with a matching /p/... page: {len(updates)}")

[MISS] https://data.biblissima.fr/entity/Q202618
[MISS] https://data.biblissima.fr/entity/Q202641
[MISS] https://data.biblissima.fr/entity/Q202515
[MISS] https://data.biblissima.fr/entity/Q202568
[MISS] https://data.biblissima.fr/entity/Q218093
[MISS] https://data.biblissima.fr/entity/Q218094
[MISS] https://data.biblissima.fr/entity/Q218095
[MISS] https://data.biblissima.fr/entity/Q218113
[MISS] https://data.biblissima.fr/entity/Q218225
[MISS] https://data.biblissima.fr/entity/Q218227
[MISS] https://data.biblissima.fr/entity/Q245334
[MISS] https://data.biblissima.fr/entity/Q244607
[MISS] https://data.biblissima.fr/entity/Q245133
[MISS] https://data.biblissima.fr/entity/Q245259
[MISS] https://data.biblissima.fr/entity/Q245132
[MISS] https://data.biblissima.fr/entity/Q259061
[MISS] https://data.biblissima.fr/entity/Q263158
[MISS] https://data.biblissima.fr/entity/Q263665
[MISS] https://data.biblissima.fr/entity/Q173849
[MISS] https://data.biblissima.fr/entity/Q261031
[OK] https://data.bi

## Save the results

In [34]:
df.loc[list(updates.keys()), "URL CoMMA"] = pd.Series(updates)

preview = df.loc[list(updates.keys()), [
    "Manuscript H-ID", 
    "bbma_data_entity", 
    "URL CoMMA"
]]

preview.to_csv("04.CoMMa\\02.metadata\\new_COMMA_links.csv", sep="\t", index=False, encoding="utf-8")